In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from more_itertools import chunked
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import cooc_smart

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config, 
    X_train, 
    X_val, 
    X_val_seq, 
    ordinal_encoder, 
    return_only_val=False , 
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 15min 33s, sys: 2min 55s, total: 18min 28s
Wall time: 3min 9s


In [5]:
%%time

list_inters = list(X_train.to_pandas()[config.col_aid])

CPU times: user 3.13 s, sys: 837 ms, total: 3.97 s
Wall time: 3.91 s


In [5]:
del X_train
gc.collect()

230

In [14]:
%%time

cooc_stats = cooc_smart.create_cooc_stats_by_chunks(train_sessions=list_inters, popularity_last_items=50, items_per_item=50, chunk_size=200_000)

creating df from sessions
creating item popularity
creating cooc_stats


100%|███████████████████████████████████████████| 67/67 [26:31<00:00, 23.76s/it]


CPU times: user 26min 19s, sys: 8min 39s, total: 34min 58s
Wall time: 35min 2s


In [15]:
%%time

pl.from_pandas(cooc_stats).write_parquet(config.cur_model_output_path + f'cooc_stats_for_val.parquet')

CPU times: user 35.8 s, sys: 11.6 s, total: 47.3 s
Wall time: 38.9 s


In [6]:
%%time

cooc_stats = pl.read_parquet(config.cur_model_output_path + f'cooc_stats_for_val.parquet').to_pandas()

CPU times: user 19.8 s, sys: 16.1 s, total: 35.9 s
Wall time: 10.4 s


In [7]:
cooc_stats.head()

,item_id,rec_item_id,cooc_count,cooc_rnk,cooc_norm
0,0,0,16,1,0.516129
1,8136,0,1,67,0.032258
2,10636,0,1,360,0.032258
3,13759,0,1,16,0.032258
4,35048,0,2,2,0.064516


In [13]:
cooc_stats[cooc_stats["item_id"] == 0].sort_values("cooc_norm", ascending=False).head()

,item_id,rec_item_id,cooc_count,cooc_rnk,cooc_norm
0,0,0,16,1,0.516129
16284,0,968176,2,29,0.400000
10334,0,35048,2,19,0.333333
89185,0,1720647,1,130,0.333333
25759,0,218900,1,51,0.333333


In [9]:
%%time

dict_cooc_stats = cooc_smart.cooc_stats_to_dict(cooc_stats, filtering_rare_keys_trs=0)

extracting max_candidates: 100%|████| 1855599/1855599 [04:11<00:00, 7380.50it/s]


CPU times: user 17min 20s, sys: 1min 53s, total: 19min 14s
Wall time: 19min 13s


In [14]:
%%time

list_inters_val = list(X_val_seq.to_pandas()[config.col_aid])

CPU times: user 205 ms, sys: 16.2 ms, total: 221 ms
Wall time: 218 ms


In [40]:
%%time

recommender = cooc_smart.CoocurenceRecommender(top_pairs=dict_cooc_stats)
recs = recommender.recommend(list_inters_val, last_k_items=5)

predicting:   0%|          | 0/466825 [00:00<?, ?it/s]

CPU times: user 2min 49s, sys: 20.8 s, total: 3min 10s
Wall time: 3min 9s


In [41]:
%%time

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(recs, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_smart_df_val_preds_{config.num_recs}.parquet')

CPU times: user 9.38 s, sys: 5.9 s, total: 15.3 s
Wall time: 6.87 s


In [42]:
%%time

recs = (
    df_val_preds.groupby(["index"])
    .agg(
        [
            pl.col("item_id").list(),
        ]
    )
    .sort("index")
    .to_pandas()
)

recs["user"] = X_val_seq.to_pandas()["user"]
recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))
recs = recs.merge(X_val[["user", "target"]].to_pandas())

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

print(utils.recall_several_targets(preds, targets))
print()
print('_' * 10)

  0%|          | 0/466825 [00:00<?, ?it/s]

0.22212249415376925

__________
CPU times: user 12.4 s, sys: 2min 14s, total: 2min 27s
Wall time: 10.5 s


In [ ]:
# 0.22212249415376925

In [4]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

CPU times: user 7min 24s, sys: 13min 4s, total: 20min 29s
Wall time: 1min 42s


0

In [7]:
%%time

cooc_stats = cooc_smart.create_cooc_stats_by_chunks(train_sessions=list_inters, popularity_last_items=50, items_per_item=50, chunk_size=200_000)

creating df from sessions
creating item popularity
creating cooc_stats


creating cooc_stats: 100%|██████████████████████| 73/73 [23:52<00:00, 19.63s/it]


CPU times: user 45min 4s, sys: 11min 14s, total: 56min 18s
Wall time: 30min 29s


In [8]:
%%time

pl.from_pandas(cooc_stats).write_parquet(config.cur_model_output_path + f'cooc_stats_for_test.parquet')

CPU times: user 36.5 s, sys: 12 s, total: 48.5 s
Wall time: 40 s


In [4]:
%%time

cooc_stats = pl.read_parquet(config.cur_model_output_path + f'cooc_stats_for_test.parquet').to_pandas()

CPU times: user 20.1 s, sys: 14.5 s, total: 34.6 s
Wall time: 12.8 s


In [5]:
%%time

dict_cooc_stats = cooc_smart.cooc_stats_to_dict(cooc_stats, filtering_rare_keys_trs=0)

transforming:   0%|          | 0/4 [00:00<?, ?it/s]

extracting max_candidates:   0%|          | 0/1855557 [00:00<?, ?it/s]

CPU times: user 16min 12s, sys: 1min 44s, total: 17min 56s
Wall time: 17min 54s


In [6]:
del cooc_stats
gc.collect()

228

In [7]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())

CPU times: user 11.1 s, sys: 1.61 s, total: 12.7 s
Wall time: 1.62 s


In [8]:
del df_test
gc.collect()

190

In [9]:
%%time

recommender = cooc_smart.CoocurenceRecommender(top_pairs=dict_cooc_stats)
recs = recommender.recommend(data_to_predict, last_k_items=5)

predicting:   0%|          | 0/1671803 [00:00<?, ?it/s]

CPU times: user 7min 59s, sys: 1min 8s, total: 9min 7s
Wall time: 9min 6s


In [11]:
%%time

df_test_preds = df_test_users.join(recs, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_smart_df_test_preds_{config.num_recs}.parquet')

CPU times: user 31.8 s, sys: 20.5 s, total: 52.3 s
Wall time: 22 s
